# Code Generator

The requirement: use a Frontier model to generate high performance C++ code from Python code


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Reminder: fetch latest code</h2>
            <span style="color:#f71;">I'm continually improving these labs, adding more examples and exercises.
            At the start of each week, it's worth checking you have the latest code.<br/>
            First do a <a href="https://chatgpt.com/share/6734e705-3270-8012-a074-421661af6ba9">git pull and merge your changes as needed</a>. Any problems? Try asking ChatGPT to clarify how to merge - or contact me!<br/><br/>
            After you've pulled the code, from the llm_engineering directory, in a Cursor Terminal, run:<br/>
            <code>uv sync</code><br/>
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Important Note</h1>
            <span style="color:#900;">
            In this lab, I use high end models GPT 5, Claude 4.5 Sonnet, Gemini 2.5 Pro, Grok 4, which are the slightly higher priced models. The costs are still low, but if you'd prefer to keep costs ultra low, please pick lower cost models like gpt-5-nano.
            </span>
        </td>
    </tr>
</table>

In [1]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import subprocess
from IPython.display import Markdown, display

In [10]:
groq_api_key = os.getenv("GROQ_API_KEY")

In [11]:
# Connect to client libraries

openai = OpenAI()

groq_url = "https://api.groq.com/openai/v1"

groq = OpenAI(api_key=groq_api_key, base_url=groq_url)

In [ ]:
OPENAI_MODEL = "gpt-5"
CLAUDE_MODEL = "claude-sonnet-4-5-20250929"
GROK_MODEL = "grok-4"
GEMINI_MODEL = "gemini-2.5-pro"

# Want to keep costs ultra-low? Uncomment these lines:

# OPENAI_MODEL = "gpt-5-nano"
# CLAUDE_MODEL = "claude-3-5-haiku-latest"
# GROK_MODEL = "grok-4-fast-non-reasoning"
# GEMINI_MODEL = "gemini-2.5-flash-lite"

In [5]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Windows',
  'arch': 'AMD64',
  'release': '11',
  'version': '10.0.26100',
  'kernel': '11',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': ''},
 'package_managers': ['winget'],
 'cpu': {'brand': '', 'cores_logical': 4, 'cores_physical': 2, 'simd': []},
 'toolchain': {'compilers': {'gcc': '', 'g++': '', 'clang': '', 'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': ''},
  'linkers': {'ld_lld': ''}}}

In [12]:
message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please reply with whether I need to install any C++ compiler to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.

System information:
{system_info}
"""

response = groq.chat.completions.create(model="openai/gpt-oss-120b", messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))
    

**Short answer:**  
Yes – you don’t have a C++ compiler installed yet. The quickest way on your Windows 11 machine is to add the free **Microsoft C++ Build Tools** (MSVC) or a lightweight **MinGW‑w64** toolchain via `winget`. After that you can compile `main.cpp` and run it from Python with a single `subprocess.run` call each.

Below are two complete “pick‑one” paths:

---

## 1️⃣ Install the Microsoft C++ Build Tools (MSVC)

These are the same compilers that ship with Visual Studio, but you only get the command‑line tools, which is about 200 MB total.

1. **Open a PowerShell (admin) window**  
   Press <kbd>Win</kbd> → type **PowerShell**, right‑click **Run as administrator**.

2. **Install the Build Tools via winget**

   ```powershell
   winget install -e --id Microsoft.VisualStudio.2022.BuildTools
   ```

   The installer will appear; just click **“Modify”** → **“Install”** and accept the defaults.  
   *Make sure the “C++ build tools” component is selected (it is by default).*

3. **Add the compiler to your PATH** (the installer does this automatically, but you may need to open a new terminal to see it).  
   Test it:

   ```powershell
   > cl /?
   Microsoft (R) C/C++ Optimizing Compiler Version ...
   ```

   If `cl` prints its help, you’re ready.

---

## 2️⃣ Install a MinGW‑w64 GCC toolchain (alternative, very lightweight)

1. **Open a PowerShell (regular user) window**.

2. **Install via winget**

   ```powershell
   winget install -e --id MinGW.MinGW64
   ```

   This pulls the “mingw‑64” package (≈120 MB) and puts `gcc`, `g++`, `ld`, etc. in `C:\Program Files\mingw64\bin`.

3. **Add the bin folder to PATH** (the installer usually does this, but verify):

   ```powershell
   > $env:Path -split ';' | Where-Object { $_ -like "*mingw64\bin*" }
   ```

   If nothing is returned, add it manually:

   ```powershell
   [Environment]::SetEnvironmentVariable(
       "Path",
       $env:Path + ";C:\Program Files\mingw64\bin",
       [EnvironmentVariableTarget]::Machine)
   ```

4. **Test the compiler**

   ```powershell
   > g++ --version
   g++.exe (Rev5, Built by MSYS2 project) 12.2.0
   ```

---

## ✅ After installation – compile & run `main.cpp` from Python

Both toolchains expose a command‑line compiler that can be invoked the same way from Python.  
Below are the exact `subprocess.run` arguments you should use for **the fastest typical release build**.

### 1. Choose the compiler command

| Toolchain | Compile command (list) | Run command (list) |
|-----------|------------------------|--------------------|
| **MSVC (`cl.exe`)** | `["cl", "/O2", "/std:c++17", "main.cpp", "/Fe:main.exe"]` | `[".\main.exe"]` |
| **MinGW/GCC (`g++.exe`)** | `["g++", "-O3", "-march=native", "-std=c++17", "main.cpp", "-o", "main.exe"]` | `[".\main.exe"]` |

*Explanation of flags*  

- **MSVC**  
  - `/O2` – full optimization.  
  - `/std:c++17` – C++17 language mode (replace with `c++20` if you need it).  
  - `/Fe:main.exe` – name of the output executable.

- **GCC/Clang**  
  - `-O3` – highest safe optimization level.  
  - `-march=native` – generate code tuned for your exact CPU (great for a single‑machine run).  
  - `-std=c++17` – language standard.  
  - `-o main.exe` – output file name.

### 2. Minimal Python snippet

```python
import subprocess
import pathlib

# Ensure we are in the folder that contains main.cpp
project_dir = pathlib.Path(__file__).parent
# (or set project_dir = pathlib.Path(r"C:\path\to\your\code"))

# -------------------------------------------------
# 1️⃣ Choose which toolchain you installed:
# -------------------------------------------------
# For MSVC (cl.exe)
compile_command = ["cl", "/O2", "/std:c++17", "main.cpp", "/Fe:main.exe"]

# For MinGW/GCC (g++.exe)
# compile_command = ["g++", "-O3", "-march=native", "-std=c++17", "main.cpp", "-o", "main.exe"]

# -------------------------------------------------
# 2️⃣ Run the compilation
# -------------------------------------------------
compile_result = subprocess.run(
    compile_command,
    cwd=project_dir,
    check=True,
    text=True,
    capture_output=True,
)

print("=== Compiler stdout ===")
print(compile_result.stdout)
print("=== Compiler stderr ===")
print(compile_result.stderr)

# -------------------------------------------------
# 3️⃣ Execute the produced binary
# -------------------------------------------------
run_command = [".\\main.exe"]   # works for both MSVC and MinGW
run_result = subprocess.run(
    run_command,
    cwd=project_dir,
    check=True,
    text=True,
    capture_output=True,
)

print("=== Program output ===")
print(run_result.stdout)
```

**What this does**

1. `subprocess.run` spawns the compiler, waits for it to finish, and raises an exception if the compilation fails (`check=True`).
2. The same function then launches the compiled `main.exe` and captures its standard output.
3. The snippet prints both the compiler’s messages and the program’s output – you can replace the `print` calls with whatever handling you need.

---

## 📌 Quick checklist

| ✅ | Item |
|---|------|
| **Compiler installed** | Run `cl` **or** `g++` from a new terminal to confirm. |
| **`main.cpp` present** | Place the file in the same folder as the Python script (or adjust `cwd`). |
| **Python 3.8+** | `subprocess.run` works on all modern Python versions. |
| **PATH correct** | The compiler executable must be reachable (`where cl` or `where g++`). |
| **Run the script** | `python compile_and_run.py` (or whatever you named it). |

That’s it – after the one‑time toolchain installation you can compile any single C++ source file with a single line of Python and get the fastest reasonable release binary for your machine. Happy coding!

## If you need to install something

If you would like to, please follow GPTs instructions! Then rerun the analysis afterwards (you might need to Restart the notebook) to confirm you're set.

You should now be equipped with the command to compile the code, and the command to run it!

Enter that in the cell below:

In [36]:
compile_command = ["clang++", "-std=c++17", "-Ofast", "-mcpu=native", "-flto=thin", "-fvisibility=hidden", "-DNDEBUG", "main.cpp", "-o", "main"]
run_command = ["./main"]

## And now, on with the main task

In [37]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [38]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [40]:
def write_output(cpp):
    with open("main.cpp", "w", encoding="utf-8") as f:
        f.write(cpp)

In [41]:
def port(client, model, python):
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    write_output(reply)

In [42]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [43]:
def run_python(code):
    globals = {"__builtins__": __builtins__}
    exec(code, globals)

In [44]:
run_python(pi)

Result: 3.141592656089
Execution Time: 57.146425 seconds


In [31]:
port(groq, "openai/gpt-oss-120b", pi)

# Compiling C++ and executing

This next cell contains the command to compile a C++ file based on the instructions from GPT.

Again, it's not crucial to do this step if you don't wish to!

OR alternatively: student Sandeep K.G. points out that you can run Python and C++ code online to test it out that way. Thank you Sandeep!  
> Not an exact comparison but you can still get the idea of performance difference.  
> For example here: https://www.programiz.com/cpp-programming/online-compiler/

In [49]:
# Use the commands from GPT 5

import subprocess
import shutil
import os
import sys
import pathlib

def compile_and_run(source='main.cpp', exe_name='main'):
    # Find a compiler on PATH
    compiler = shutil.which('clang++') or shutil.which('g++') or shutil.which('cl')
    if not compiler:
        raise FileNotFoundError(
            "No C++ compiler found on PATH. Install clang (LLVM) or g++ (MinGW/MSYS/WSL) "
            "or Microsoft Visual C++ (cl) and add it to your PATH."
        )

    compiler_base = os.path.basename(compiler).lower()
    is_windows = (os.name == 'nt')

    # Decide output executable name for platform
    out_name = exe_name + ('.exe' if is_windows else '')
    out_path = pathlib.Path(out_name)

    # Compose compile command depending on compiler
    if 'cl' in compiler_base:  # MSVC cl
        # Simple invocation for cl; adjust flags as needed
        compile_cmd = [compiler, '/EHsc', source, '/Fe:' + str(out_name)]
    else:  # clang++ or g++
        # Remove flags unsupported on some platforms if necessary. They may be ignored or error out.
        compile_cmd = [
            compiler,
            "-std=c++17",
            "-Ofast",
            "-mcpu=native",
            "-flto=thin",
            "-fvisibility=hidden",
            "-DNDEBUG",
            source,
            "-o", str(out_name)
        ]

    # Try to compile and show helpful errors
    try:
        print("Compiling with:", " ".join(compile_cmd))
        cp = subprocess.run(compile_cmd, check=True, text=True, capture_output=True)
        if cp.stdout:
            print("compiler stdout:", cp.stdout)
        if cp.stderr:
            print("compiler stderr:", cp.stderr, file=sys.stderr)
    except FileNotFoundError:
        raise FileNotFoundError(f"Compiler executable not found: {compiler}")
    except subprocess.CalledProcessError as e:
        # Show details to help debugging
        print("Compilation failed.")
        if e.stdout:
            print("compiler stdout:\n", e.stdout)
        if e.stderr:
            print("compiler stderr:\n", e.stderr, file=sys.stderr)
        raise

    # Ensure output binary exists
    if not out_path.exists():
        raise FileNotFoundError(f"Expected output '{out_name}' not found after compilation.")

    # Run the executable three times
    if is_windows:
        run_cmd = [str(out_name)]          # Windows: "main.exe"
    else:
        run_cmd = ['./' + str(out_name)]  # Unix: "./main"

    for i in range(1, 4):
        try:
            print(f"\n--- Run #{i} ---")
            r = subprocess.run(run_cmd, check=True, text=True, capture_output=True)
            print(r.stdout, end='')
            if r.stderr:
                print("program stderr:\n", r.stderr, file=sys.stderr)
        except FileNotFoundError:
            raise FileNotFoundError(f"Could not execute {run_cmd}. Check file exists and is executable.")
        except subprocess.CalledProcessError as e:
            print("Program returned non-zero exit code.")
            if e.stdout:
                print("program stdout:\n", e.stdout)
            if e.stderr:
                print("program stderr:\n", e.stderr, file=sys.stderr)
            raise

# # Example usage
# if __name__ == '__main__':
#     compile_and_run()


In [ ]:
compile_and_run()

In [ ]:
19.178207/0.082168

In [51]:
print(f"""
In Ed's experiments, the performance speedups were:

4th place: Claude Sonnet 4.5: {19.178207/0.104241:.0f}X speedup
3rd place: GPT-5: {19.178207/0.082168:.0f}X speedup
2nd place: Grok 4: {19.178207/0.018092:.0f}X speedup
1st place: Gemini 2.5 Pro: {19.178207/0.013314:.0f}X speedup
""")


In Ed's experiments, the performance speedups were:

4th place: Claude Sonnet 4.5: 184X speedup
3rd place: GPT-5: 233X speedup
2nd place: Grok 4: 1060X speedup
1st place: Gemini 2.5 Pro: 1440X speedup

